In [1]:
%pip install --upgrade pip
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 4.5 MB/s eta 0:00:01
   -------------------------- ------------- 1.3/2.0 MB 3.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 3.8 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
   ---------------------------------------- 0.0/2.5 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.5 GB 5.5 MB/s eta 0:07:39
   ---------------------------------------- 0.0/2.5 GB 5.1 MB/s eta 0:08:15
   ---------------------------------------- 0.0/2.5 GB 5.2 MB/s eta 0:08:07
   ---------------------------------------- 0.0/2.5 GB 5.1 MB/s eta 0:08:11
   ---------------------------------------- 0.0/2.5 GB 5.2 MB/s eta 0:0

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

batch_size = 128
use_cuda = device.type == "cuda"

transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

trainset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root="./data", train=False, download=True, transform=transform_test
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=use_cuda,
)
testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=use_cuda,
)

Using device: cuda


100.0%


In [3]:
class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_planes,
                    planes * self.expansion,
                    kernel_size=1,
                    stride=stride,
                    bias=False,
                ),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + self.shortcut(x)
        out = self.relu(out)
        return out


class ResNetCIFAR(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_planes = 16

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64 * block.expansion, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out


def resnet56(num_classes=10):
    return ResNetCIFAR(BasicBlock, [9, 9, 9], num_classes=num_classes)


model = resnet56().to(device)

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4
)

epochs = 160
scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[82, 123], gamma=0.1)

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in trainloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    scheduler.step()
    epoch_loss = running_loss / len(trainset)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        current_lr = optimizer.param_groups[0]["lr"]
        print(
            f"Epoch [{epoch + 1}/{epochs}] | Loss: {epoch_loss:.4f} | LR: {current_lr}"
        )

Epoch [1/160] | Loss: 1.8762 | LR: 0.1
Epoch [10/160] | Loss: 0.5132 | LR: 0.1
Epoch [20/160] | Loss: 0.3279 | LR: 0.1
Epoch [30/160] | Loss: 0.2564 | LR: 0.1
Epoch [40/160] | Loss: 0.2206 | LR: 0.1
Epoch [50/160] | Loss: 0.1996 | LR: 0.1
Epoch [60/160] | Loss: 0.1890 | LR: 0.1
Epoch [70/160] | Loss: 0.1746 | LR: 0.1
Epoch [80/160] | Loss: 0.1695 | LR: 0.1
Epoch [90/160] | Loss: 0.0276 | LR: 0.010000000000000002
Epoch [100/160] | Loss: 0.0142 | LR: 0.010000000000000002
Epoch [110/160] | Loss: 0.0105 | LR: 0.010000000000000002
Epoch [120/160] | Loss: 0.0075 | LR: 0.010000000000000002
Epoch [130/160] | Loss: 0.0055 | LR: 0.0010000000000000002
Epoch [140/160] | Loss: 0.0048 | LR: 0.0010000000000000002
Epoch [150/160] | Loss: 0.0048 | LR: 0.0010000000000000002
Epoch [160/160] | Loss: 0.0042 | LR: 0.0010000000000000002


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100.0 * correct / total
print(f"Test Accuracy on {total} images: {accuracy:.2f}%")

torch.save(model.state_dict(), "resnet56_cifar10.pt")

Test Accuracy on 10000 images: 92.91%
